# 09 — How deep, not just whether (Phase 4)

A binary alert tells an operations room that a road will flood. It does not tell
them whether to expect 6 cm of standing water or 40 cm that will strand a bus.
Those call for different responses, and config maps depth directly to CAP
severity:

```
alerting.warning_p95_depth_cm:  {1h: 30, 3h: 26, 6h: 24}
```

So this notebook predicts **depth**, and specifically the upper end of it.

### Why quantiles and not an average

Flood depth is zero in about 99% of rows. The *mean* of that distribution is a
couple of millimetres — a number no one can act on, and one that is wrong in both
directions: far too high for a dry road, far too low for a flooded one.

Quantile regression predicts a level the water will stay below with a chosen
probability. `alpha=0.95` gives the p95 depth used for CAP severity. Pairing
0.05 and 0.95 gives the 90% interval that `quantile_coverage_target` asks for.

### The part that must not be skipped

**Nominal coverage is a claim, not a fact.** A model asked for the 95th
percentile does not necessarily deliver it on skewed data — it can be badly
optimistic exactly where it matters, on the deep tail. Part 3 measures the
coverage actually achieved on the test year and compares it against the tolerance
band in config. If it misses, the number must not be published as a p95.

### What you need

Notebooks 05–08. About 8 minutes.

## Setup

In [1]:
import os, sys, json, time, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

_here = Path.cwd()
_root = next(p for p in [_here, *_here.parents] if (p / "config/config.yaml").is_file())
sys.path.insert(0, str(_root / "src"))
os.chdir(_root)

import numpy as np
import pandas as pd
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 80)

from bkkflood.config import load_config
from bkkflood.rawio import connect
from bkkflood.evaluate import folds
from bkkflood.models import run_fold, gain_importance

CFG = load_config()
FOLDS = folds()
TIERS = sorted(int(t) for t in CFG["flood_event"]["tiers_cm"].values())
HORIZONS = CFG["horizons_hours"]
BASE = pd.read_parquet("data/features/baseline_results.parquet")
con = connect()
print("folds:", [f["test"] for f in FOLDS], "| tiers:", TIERS, "| horizons:", HORIZONS)
from bkkflood.models import build_matrix, train_quantile, score_year
QT = CFG["objective"]["quantile_coverage_target"]
QTOL = CFG["objective"]["quantile_coverage_tolerance"]
WARN = CFG["alerting"]["warning_p95_depth_cm"]
print(f"coverage target {QT}, tolerance {QTOL} | CAP warning depths {WARN}")

folds: [2022, 2023, 2024, 2025] | tiers: [5, 15, 30] | horizons: [1, 3, 6]
coverage target 0.9, tolerance [0.85, 0.93] | CAP warning depths {1: 30, 3: 26, 6: 24}


## 1. Train the quantile models

Two per horizon: the 5th and 95th percentile of the deepest water in the forward
window. Trained on the same rows and the same features as the general model —
only the question changes, from *will it* to *how deep*.

In [2]:
ALPHAS = [round((1 - QT) / 2, 3), round(1 - (1 - QT) / 2, 3)]   # e.g. 0.05, 0.95
print("quantiles:", ALPHAS)

models, rows = {}, []
t0 = time.time()
for fold in FOLDS:
    for h in HORIZONS:
        X, y, meta = build_matrix(fold["train"], 15, h, "general", 0.05,
                                  con=con, target="depth")
        Xv, yv, _ = build_matrix([fold["val"]], 15, h, "general", 0.05, con=con,
                                 columns=meta["features"], target="depth")
        ok, okv = ~np.isnan(y), ~np.isnan(yv)
        for a in ALPHAS:
            b = train_quantile(X[ok], y[ok], Xv[okv], yv[okv], alpha=a)
            models[(fold["test"], h, a)] = (b, meta["features"])
            rows.append({"test_year": fold["test"], "horizon_h": h, "alpha": a,
                         "train_rows": int(ok.sum()),
                         "best_iteration": int(b.best_iteration or 0)})
    print(f"fold tested on {fold['test']} done  ({time.time()-t0:.0f}s)")
QMODELS = pd.DataFrame(rows)
print("\n", QMODELS.shape)

quantiles: [0.05, 0.95]
fold tested on 2022 done  (29s)
fold tested on 2023 done  (67s)
fold tested on 2024 done  (121s)
fold tested on 2025 done  (179s)

 (24, 5)


## 2. Predict depth on the test years

Predictions are clipped at zero. A negative depth is not a cautious estimate, it
is a nonsense one, and it would flow straight into a CAP message.

In [3]:
# Both quantile models share the same feature matrix, so the year is read
# once and predicted twice. Reading it per model doubles the I/O for no
# reason, and a year is 3.7 million rows.
from bkkflood.config import resolve

def depth_frame(test_year, h, chunk=750_000):
    lo_b, feats = models[(test_year, h, ALPHAS[0])]
    hi_b, _ = models[(test_year, h, ALPHAS[1])]
    path = str(resolve(f"data/features/features_{test_year}.parquet"))
    keep = ["station_code", "ts", "fl_depth_now", f"y_maxdepth_{h}h"]
    sel = ", ".join([f'"{c}"' for c in feats] + [f'"{c}"' for c in keep])
    n = con.execute(f"SELECT count(*) FROM '{path}' WHERE y_valid_{h}h").fetchone()[0]
    out = []
    for off in range(0, n, chunk):
        d = con.execute(f"SELECT {sel} FROM '{path}' WHERE y_valid_{h}h "
                        f"LIMIT {chunk} OFFSET {off}").fetchdf()
        if d.empty:
            break
        Xc = d[feats]
        r = d[keep].copy()
        r["q_lo"] = np.clip(lo_b.predict(Xc, num_iteration=lo_b.best_iteration), 0, None)
        r["q_hi"] = np.clip(hi_b.predict(Xc, num_iteration=hi_b.best_iteration), 0, None)
        out.append(r)
    return pd.concat(out, ignore_index=True)

DEPTH = {}
for fold in FOLDS:
    for h in HORIZONS:
        DEPTH[(fold["test"], h)] = depth_frame(fold["test"], h)
    print(f"scored {fold['test']}")
print("done")

scored 2022
scored 2023
scored 2024
scored 2025
done


## 3. Does the interval actually cover what it claims?

The check that decides whether any of this is publishable.

`coverage` is the fraction of test rows whose true depth fell inside the
predicted interval. It should land near the target. Two failure modes, and they
are not symmetric:

- **coverage far below target** — the interval is too narrow. The model is
  overconfident and CAP messages will understate real floods. This is the
  dangerous direction.
- **coverage far above target** — the interval is too wide and says nothing
  useful, but it does not mislead.

Coverage is also reported on **wet rows only**. Overall coverage is easy: predict
`[0, 0]` everywhere and be right 99% of the time. The wet-row number is the one
with any content.

In [4]:
rows = []
for (yr, h), d in DEPTH.items():
    truth = d[f"y_maxdepth_{h}h"]
    ok = truth.notna()
    inside = (truth >= d.q_lo - 1e-9) & (truth <= d.q_hi + 1e-9)
    wet = ok & (truth > 0)
    rows.append({
        "test_year": yr, "horizon_h": h,
        "rows": int(ok.sum()),
        "coverage_all": float(inside[ok].mean()),
        "wet_rows": int(wet.sum()),
        "coverage_wet": float(inside[wet].mean()) if wet.any() else np.nan,
        "median_interval_cm": float((d.q_hi - d.q_lo)[ok].median()),
        "median_q_hi_wet": float(d.q_hi[wet].median()) if wet.any() else np.nan,
    })
COV = pd.DataFrame(rows)
print(f"target {QT}, acceptable band {QTOL}\n")
print(COV.groupby("horizon_h")[["rows", "coverage_all", "wet_rows",
                                "coverage_wet", "median_interval_cm",
                                "median_q_hi_wet"]].mean().round(4).to_string())

target 0.9, acceptable band [0.85, 0.93]

                 rows  coverage_all  wet_rows  coverage_wet  median_interval_cm  median_q_hi_wet
horizon_h                                                                                       
1          3409638.75        0.9989   8244.75        0.6291              0.0000           8.4624
3          3409256.50        0.9972  15114.25        0.4325              0.0000           3.1079
6          3408681.50        0.9956  25009.00        0.4404              0.0057           4.3243


In [5]:
lo, hi = QTOL
summary = COV.groupby("horizon_h")[["coverage_all", "coverage_wet"]].mean()
print("verdict per horizon\n")
for h, r in summary.iterrows():
    a_ok = lo <= r.coverage_all <= hi
    w_ok = lo <= r.coverage_wet <= hi
    print(f"  {h}h   all rows {r.coverage_all:.3f} {'OK ' if a_ok else 'OUT'}"
          f"   wet rows {r.coverage_wet:.3f} {'OK ' if w_ok else 'OUT'}")

print()
if not summary.coverage_wet.between(lo, hi).all():
    print("At least one horizon misses the band ON WET ROWS. Do not publish those")
    print("intervals as a p95 — either widen the quantiles, train the depth model")
    print("on wet rows only, or report depth as a range with a stated caveat.")
    print("An interval that claims 90% and delivers less is worse than no interval.")
else:
    print("Coverage is inside the configured band. The p95 is usable for CAP.")

verdict per horizon

  1h   all rows 0.999 OUT   wet rows 0.629 OUT
  3h   all rows 0.997 OUT   wet rows 0.432 OUT
  6h   all rows 0.996 OUT   wet rows 0.440 OUT

At least one horizon misses the band ON WET ROWS. Do not publish those
intervals as a p95 — either widen the quantiles, train the depth model
on wet rows only, or report depth as a range with a stated caveat.
An interval that claims 90% and delivers less is worse than no interval.


## 4. The number CAP actually consumes

`alerting.warning_p95_depth_cm` sets the depth at which an alert is escalated to
a warning. Here is how often the predicted p95 would cross it, and how often the
real water did.

A large gap in either direction is a calibration problem that must be settled
before Phase 10 generates a single message.

In [6]:
rows = []
for (yr, h), d in DEPTH.items():
    thr = WARN.get(h) or WARN.get(str(h))
    truth = d[f"y_maxdepth_{h}h"]
    rows.append({
        "test_year": yr, "horizon_h": h, "cap_threshold_cm": thr,
        "predicted_warnings": int((d.q_hi >= thr).sum()),
        "actual_exceedances": int((truth >= thr).sum()),
        "caught": int(((d.q_hi >= thr) & (truth >= thr)).sum()),
    })
CAP = pd.DataFrame(rows)
CAP["recall_of_deep_floods"] = CAP.caught / CAP.actual_exceedances.replace(0, np.nan)
CAP["alerts_per_true_event"] = CAP.predicted_warnings / CAP.actual_exceedances.replace(0, np.nan)
print(CAP.groupby("horizon_h")[["cap_threshold_cm", "predicted_warnings",
                                "actual_exceedances", "recall_of_deep_floods",
                                "alerts_per_true_event"]].mean().round(3).to_string())
print()
print("alerts_per_true_event is the operational cost: how many warnings BMA would")
print("issue for each genuinely deep flood. It is a staffing number, not a metric.")

           cap_threshold_cm  predicted_warnings  actual_exceedances  recall_of_deep_floods  alerts_per_true_event
horizon_h                                                                                                        
1                      30.0              584.75              212.75                  0.664                  3.273
3                      26.0             2486.75              652.75                  0.382                  4.570
6                      24.0             6275.50             1425.25                  0.239                  6.336

alerts_per_true_event is the operational cost: how many warnings BMA would
issue for each genuinely deep flood. It is a staffing number, not a metric.


In [7]:
out = Path("docs/reports/model_depth_quantiles.md")
with out.open("w") as f:
    f.write("# Depth quantiles\n\n")
    f.write("Generated by `notebooks/09_train_depth_quantiles.ipynb`.\n\n")
    f.write(f"Quantiles {ALPHAS}, coverage target {QT}, tolerance {QTOL}.\n\n")
    f.write("## Coverage achieved on the test years\n\n")
    f.write(COV.groupby("horizon_h")[["coverage_all", "coverage_wet",
                                      "median_interval_cm"]].mean().round(4).to_markdown())
    f.write("\n\nCoverage on **wet rows** is the meaningful figure. Overall coverage is\n")
    f.write("trivially high because 99% of rows are dry and `[0, 0]` covers them.\n\n")
    f.write("## Against the CAP escalation thresholds\n\n")
    f.write(CAP.groupby("horizon_h")[["cap_threshold_cm", "predicted_warnings",
                                      "actual_exceedances", "recall_of_deep_floods",
                                      "alerts_per_true_event"]].mean().round(3).to_markdown())
    f.write("\n")
print("wrote", out)
COV.to_parquet("data/features/depth_coverage.parquet", index=False)

wrote docs/reports/model_depth_quantiles.md


## Phase 4 is complete

| Notebook | Model | Question |
|---|---|---|
| 07 | general | will this station reach the tier? |
| 08 | **onset** | **this road is dry — will it flood?** |
| 09 | depth quantiles | how deep, with an interval |

**Next: Phase 5, honest evaluation.** Calibration, error analysis, and the
report that goes to BMA — including the false negatives, which are the floods
nobody was warned about.

Two things must be carried into it rather than quietly dropped:

1. **`era5_*` is excluded from every model.** It is reanalysis, published about
   five days late, so it exists in training and not at serving time. It was worth
   a fifth of the onset model's PR-AUC and was dropped anyway.
2. **`rain_fcst_*` reaches the onset model only**, and the Phase 3 measurement
   behind that decision was marginal. Re-run the ablation properly in Phase 5.